# Chatbot from Scratch

Preprocessing:
- Lemmatisation
-



Design:
- Intentrecogintions
- Named Entity Recognition
- Diaglog Flow builder


## Dataset

In [ ]:
import pandas as pd
path = r"C:\Users\ricca\PycharmProjects\nlp-projects\src\skitest.txt"
with open(path,encoding = 'UTF-8') as f:
    lines = f.readlines()

In [ ]:
import spacy
nlp = spacy.load("de_core_news_sm")

In [ ]:
docs = nlp(lines[0])
for token in docs:
    print(token.text, token.lemma_, token.pos_, token.tag_, token.dep_,
            token.shape_, token.is_alpha, token.is_stop)

In [ ]:
dok = u"Google hat sein Hauptquartier in Mountain View, Kaliforniern"
dok = nlp(dok)
for token in dok.ents:
    print(token.text,token.label_)

In [ ]:
from intentclassifier import intentclassifier
inte = intentclassifier()
inte.intent_preprocessing(dok)

In [ ]:
from transformers import BertModel, BertConfig

# Initializing a BERT bert-base-uncased style configuration

configuration = BertConfig()

# Initializing a model from the bert-base-uncased style configuration

model = BertModel(configuration)

# Accessing the model configuration

configuration = model.config

In [ ]:
import os
modelInputFolder = '~/src/'
bert_model_name="uncased_L-12_H-768_A-12"
bert_ckpt_dir = os.path.join(modelInputFolder, bert_model_name)
bert_ckpt_file = os.path.join(bert_ckpt_dir, "bert_model.ckpt")
bert_config_file = os.path.join(bert_ckpt_dir, "bert_config.json")
print(bert_ckpt_dir)
print(bert_ckpt_file)
print(bert_config_file)

In [ ]:
from bert.tokenization.bert_tokenization import FullTokenizer

In [ ]:
tokenizer = FullTokenizer(vocab_file)
print(tokenizer)

In [ ]:
tokens = tokenizer.tokenize("Hello, How are you?")
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(tokens)
print(token_ids)

## Hugging Face

In [ ]:
from datasets import load_dataset

dataset = load_dataset("yelp_review_full")

dataset["train"][100]

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)


tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [ ]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=5)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(output_dir="test_trainer")

In [ ]:
import numpy as np
from datasets import load_metric

metric = load_metric("accuracy")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(output_dir="test_trainer", evaluation_strategy="epoch")

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()